# 🚀 Notebook do Professor (Demo) — Aula 08: Interfaces com Gradio e Streamlit + deploy com URL pública

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 08/14 — Módulo 3: Interfaces, Agentes e Integração**  
**⏱️ 1h40min**  
**🌐 Gradio · ngrok · streaming**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Ao final da aula, o RAG do CKP02 está acessível via URL pública — qualquer pessoa com o link consegue fazer perguntas aos documentos do domínio pelo celular ou computador, sem abrir o Colab.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — Gradio — quando usar cada componente

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
!pip install gradio -q

import gradio as gr

# gr.Interface — a versão mais simples
def responder(pergunta: str) -> str:
    """Qualquer função Python vira uma interface."""
    return chain_rag.invoke(pergunta)

demo = gr.Interface(
    fn=responder,
    inputs=gr.Textbox(label="Pergunta", placeholder="Digite aqui..."),
    outputs=gr.Textbox(label="Resposta"),
    title="DocMind RAG",
    description="Pergunte sobre os documentos do domínio",
)
demo.launch()  # abre interface no Colab inline ou em localhost

### Slide 08 — gr.ChatInterface — chatbot com histórico automático

In [ ]:
import gradio as gr

# A função de chat recebe mensagem + histórico
def chat_rag(mensagem: str, historico: list) -> str:
    """Recebe a pergunta atual e o histórico [{role, content}]."""
    # Por enquanto RAG stateless — Slide 13 adiciona memória com RunnableWithMessageHistory
    resposta = chain_rag.invoke(mensagem)
    return resposta

# ChatInterface detecta automaticamente: função recebe (mensagem, historico) → str
chatbot = gr.ChatInterface(
    fn=chat_rag,
    title="DocMind — Assistente do Domínio",
    description="Pergunte qualquer coisa sobre os documentos do grupo.",
    examples=[
        "Qual é o prazo de entrega mencionado no manual?",
        "Como acionar o suporte técnico?",
        "Quais são as cláusulas de garantia?",
    ],
    chatbot=gr.Chatbot(height=400),
    type="messages",  # formato OpenAI: [{role, content}]
)
chatbot.launch()

### Slide 09 — Streaming com yield — respostas token a token

In [ ]:
def chat(msg, hist):
    # Espera TUDO antes de retornar
    resp = chain_rag.invoke(msg)
    return resp  # 5-30 segundos

# Usuário: ⏳ (espera 15s) → resposta aparece

### Slide 09 — Streaming com yield — respostas token a token

In [ ]:
def chat_stream(msg, hist):
    parcial = ""
    # .stream() em vez de .invoke()
    for chunk in chain_rag.stream(msg):
        parcial += chunk
        yield parcial  # envia a cada token

# Usuário: "Conf" → "Confor" → "Conforme..."

### Slide 09 — Streaming com yield — respostas token a token

In [ ]:
def chat_rag_stream(mensagem: str, historico: list):
    """Gerador: yield parcial acumulado a cada chunk."""
    parcial = ""
    for chunk in chain_rag.stream(mensagem):
        parcial += chunk  # acumula — usuário vê o texto crescer
        yield parcial

# ChatInterface detecta yield automaticamente e ativa streaming
chatbot = gr.ChatInterface(fn=chat_rag_stream, type="messages",
                             title="DocMind RAG — Streaming")
chatbot.launch()

### Slide 11 — ngrok — URL pública em 3 linhas

In [ ]:
!pip install gradio pyngrok -q

from pyngrok import ngrok
from google.colab import userdata
import gradio as gr

# Autenticar o ngrok (gratuito — token no Colab Secrets)
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

# Opção A — share=True (mais simples, usa infra do Gradio HuggingFace)
chatbot.launch(share=True)
# → Imprime: "Running on public URL: https://abc123.gradio.live"

# Opção B — ngrok explícito (mais controle, URL estável por sessão)
porta = 7860
tunel = ngrok.connect(porta)
print(f"URL pública: {tunel.public_url}")
chatbot.launch(server_port=porta, server_name="0.0.0.0")
# → URL válida enquanto a sessão do Colab estiver ativa

### Slide 13 — RunnableWithMessageHistory — RAG com memória multi-sessão

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Store em memória — uma entrada por session_id
store: dict = {}

def obter_historico(session_id: str) -> ChatMessageHistory:
    """Retorna (ou cria) o histórico do usuário pelo session_id."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Prompt RAG com histórico (adiciona {chat_history} ao prompt)
PROMPT_RAG_HIST = """<persona>Assistente do domínio com memória de conversa.</persona>
<historico>{chat_history}</historico>
<contexto>{contexto}</contexto>
<instrucoes>Use o contexto para responder. Cite fontes. Se a pergunta se refere ao histórico, use-o.</instrucoes>
<pergunta>{pergunta}</pergunta>"""

chain_com_hist = (
    {
        "contexto":     retriever | RunnableLambda(formatar_contexto),
        "pergunta":     RunnablePassthrough(),
        "chat_history": RunnableLambda(lambda _: ""),  # preenchido pelo wrapper
    }
    | ChatPromptTemplate.from_template(PROMPT_RAG_HIST)
    | llm | StrOutputParser()
)

chain_rag_memoria = RunnableWithMessageHistory(
    chain_com_hist,
    obter_historico,
    input_messages_key="pergunta",
    history_messages_key="chat_history",
)

### Slide 14 — Gradio + RunnableWithMessageHistory — chatbot completo

In [ ]:
import uuid, gradio as gr

def chat_com_memoria(mensagem: str, historico: list, session_id: str):
    """Streaming com memória por session_id."""
    parcial = ""
    for chunk in chain_rag_memoria.stream(
        {"pergunta": mensagem},
        config={"configurable": {"session_id": session_id}},  # chave do histórico
    ):
        parcial += chunk
        yield parcial

with gr.Blocks(title="DocMind RAG") as demo:
    # session_id gerado automaticamente por sessão de browser
    session_id = gr.State(lambda: str(uuid.uuid4()))

    gr.Markdown("## 📄 DocMind — Assistente do Domínio")
    chatbot_ui = gr.ChatInterface(
        fn=chat_com_memoria,
        type="messages",
        additional_inputs=[session_id],  # passa o session_id como input extra
        examples=["Qual é o prazo de garantia?", "Como acionar o suporte?"],
    )

demo.launch(share=True)  # → URL pública gerada automaticamente

### Slide 15 — Streamlit — alternativa com st.session_state

In [ ]:
!pip install streamlit -q  # rodar como app separado: !streamlit run app.py &

import streamlit as st

st.title("DocMind — Assistente do Domínio")

# Inicializar histórico no session_state (persiste entre re-renders)
if "mensagens" not in st.session_state:
    st.session_state.mensagens = []

# Exibir mensagens anteriores
for msg in st.session_state.mensagens:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Input do usuário
if pergunta := st.chat_input("Faça sua pergunta..."):
    st.session_state.mensagens.append({"role":"user", "content":pergunta})
    with st.chat_message("user"):
        st.write(pergunta)
    with st.chat_message("assistant"):
        resposta = st.write_stream(chain_rag.stream(pergunta))  # streaming nativo
    st.session_state.mensagens.append({"role":"assistant", "content":resposta})

### Slide 22 — Python novo desta aula

```
# 1. yield — transforma função em gerador (streaming)
def gerar_numeros():
    for i in range(5):
        yield i        # pausa aqui, retorna i, retoma na próxima iteração
list(gerar_numeros())  # → [0, 1, 2, 3, 4]

# 2. Walrus operator := — atribuição dentro de if
if pergunta := st.chat_input("..."):  # atribui E testa ao mesmo tempo
    print(pergunta)  # só entra se pergunta não for None/vazio

# 3. lambda sem argumento — gr.State usa para valor inicial por sessão
uuid_factory = lambda: str(uuid.uuid4())  # chamada a cada nova sessão
estado       = gr.State(uuid_factory)       # sem () — passa a função, não o resultado

# 4. += em string (acumular streaming)
parcial = ""
for chunk in ["Conf", "orme", " a p", "ágina"]:
    parcial += chunk   # acumula: "Conf" → "Conforme" → "Conforme a p" → ...
    yield parcial      # envia o estado acumulado a cada passo

# 5. Dict aninhado como config do LangChain
config = {"configurable": {"session_id": meu_session_id}}
chain.stream(input, config=config)  # config é kw-only, sempre nomeado
```

### Slide 26 — 🚀 Demo Prática ao Vivo — RAG do grupo em 15 linhas extras

In [ ]:
!pip install gradio -q  # já deve estar instalado

import gradio as gr, uuid

# ── Memória por sessão ──────────────────────────────────────────────
store = {}
def obter_hist(sid):
    if sid not in store: store[sid] = ChatMessageHistory()
    return store[sid]

# ── Chat com streaming + memória ────────────────────────────────────
def chat(msg, hist, sid):
    parcial = ""
    for chunk in chain_rag_memoria.stream(
        {"pergunta": msg},
        config={"configurable": {"session_id": sid}},
    ):
        parcial += chunk
        yield parcial

# ── Interface ────────────────────────────────────────────────────────
with gr.Blocks(title="DocMind RAG") as demo:
    sid  = gr.State(lambda: str(uuid.uuid4()))
    gr.Markdown("# 📄 DocMind — Assistente do Domínio\nPergunte sobre os documentos do grupo.")
    gr.ChatInterface(fn=chat, type="messages", additional_inputs=[sid])

demo.launch(share=True)
# → "Running on public URL: https://abc123.gradio.live"
# → professor projeta o QR Code — alunos acessam pelo celular

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 08 · 2º Semestre**  
### Publicar a primeira URL ao vivo do grupo ★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 4 lacunas — retriever, chain com memória (input/history keys), função de chat com streaming e lançamento com share=True.
2. Personalize a interface — título do grupo, 3 exemplos de perguntas reais do domínio como botões de atalho.
3. Publique e compartilhe — copie a URL gerada e poste no grupo da turma. Acesse a URL de outro grupo e faça 2 perguntas.
4. Documente: a interface lembrou o contexto da primeira pergunta na segunda? Teste perguntas de acompanhamento tipo "e sobre isso que você disse..."

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: "k": 3
>
> Lacuna 2: RunnablePassthrough() para "pergunta"; input_messages_key="pergunta"; history_messages_key="chat_history"
>
> Lacuna 3: msg para "pergunta"; sid para "session_id" no config
>
> Lacuna 4: chat para fn; sid para additional_inputs; True para share

In [ ]:
!pip install gradio langchain-community langchain-ollama chromadb pymupdf -q

import gradio as gr, uuid, os
from google.colab import userdata
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Reutilizar db do CKP02 (já indexado — só recarregar)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db         = Chroma(persist_directory="/content/ckp02", embedding_function=embeddings)

# 👉 LACUNA 1: crie o retriever com k=3
retriever = db.as_retriever(search_kwargs={"k":___})

# 👉 LACUNA 2: monte a chain RAG com memória
store = {}
def obter_hist(sid):
    if sid not in store: store[sid] = ChatMessageHistory()
    return store[sid]

chain_base = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "chat_history": RunnableLambda(lambda _:"")}
    | prompt_com_hist | llm | StrOutputParser()
)
chain_mem = RunnableWithMessageHistory(
    chain_base, obter_hist,
    input_messages_key=___,
    history_messages_key=___,
)

# 👉 LACUNA 3: função de chat com streaming
def chat(msg, hist, sid):
    parcial = ""
    for chunk in chain_mem.stream({"pergunta":___},
                                   config={"configurable":{"session_id":___}}):
        parcial += chunk
        yield parcial

# 👉 LACUNA 4: monte a interface e publique com share=True
with gr.Blocks() as demo:
    sid = gr.State(lambda:str(uuid.uuid4()))
    gr.Markdown("## 📄 DocMind do Grupo — [Nome do Domínio]")
    gr.ChatInterface(fn=___, type="messages", additional_inputs=[___])
demo.launch(share=___)  # True para URL pública

## 📚 Referências da aula

- Docs Gradio — ChatInterface, Blocks, streaming e deploy. gradio.app/docs/gradio/chatinterface
- Docs LangChain — RunnableWithMessageHistory para múltiplas sessões. python.langchain.com/docs/how_to/message_history
- Docs ngrok — Túnel HTTP gratuito para desenvolvimento e demos. ngrok.com/docs
- Docs Streamlit — st.chat_message, st.session_state, file_uploader. docs.streamlit.io/develop/api-reference/chat
- Livro Avila, R. D. — Architecting AI Software Systems. Packt, 2025. Cap. 5 — o padrão "Blue and Gold" para deploy seguro de pipelines de IA em produção.
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes inteligentes: a fundamentação teórica da transição de pipeline para agente que acontece nas próximas aulas.

---

**Próxima Aula — Aula 09** — Agentes de IA — ReAct, tools e function calling
  
O chatbot passa a decidir qual ferramenta usar. RAG, web search e calculadora — autonomamente.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*